In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
dataset_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(dataset_url)

In [5]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [9]:
df.shape

(5000, 9)

In [11]:
# Data preparation
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [13]:
categorical_columns = [
    "lead_source",
    "industry",
    "employment_status",
    "location" 
]
    
numerical_columns = [
    "annual_income",
    "lead_score"
]

df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [15]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [17]:
# Question 1: Mode for industry

df["industry"].mode()

0    technology
Name: industry, dtype: object

In [19]:
# Question 2: Biggest correlation

numerical_features = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score"  
]

correlation_matrix = df[numerical_features].corr()

feature_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

for feature_1, feature_2 in feature_pairs:
    correlation = correlation_matrix.loc[feature_1, feature_2]
    print(feature_1, "and", feature_2, ":", correlation)

interaction_count and lead_score : 0.9157457980418697
number_of_courses_viewed and lead_score : 0.7572042392104886
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.12284235135955958


In [21]:
# Split the data

df_full_train, df_test = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, 
    test_size=0.25, 
    random_state=42
)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [23]:
# Remove converted from dataframe

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [25]:
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='object')

In [29]:
# Question 3: Biggest Mutual Information

def calculate_mutual_information_score(series):
    return round(mutual_info_score(series, y_train), 2)

mi = df_train[categorical_columns].apply(
        calculate_mutual_information_score
    )

mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [37]:
# Question 4: Accuracy

# Use one-hot encoding
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical_columns + numerical_features].to_dict(orient="records")
val_dicts = df_val[categorical_columns + numerical_features].to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train the logistic regression model
model = LogisticRegression(
    solver='liblinear', 
    C=1.0, 
    max_iter=1000, 
    random_state=42
)

model.fit(X_train, y_train)

# Predict on the valication set
y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()

accuracy

0.645

In [39]:
# Question 5: Feature selection

feature_columns = categorical_columns + numerical_features

def get_accuracy(features):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    return (y_pred == y_val).mean()


original_accuracy = get_accuracy(feature_columns)

print(original_accuracy)

0.645


In [41]:
for feature in feature_columns:
    reduced_features = [
        col for col in feature_columns
        if col != feature
    ]

    accuracy_without = get_accuracy(reduced_features)

    difference = original_accuracy - accuracy_without

    print(
        feature,
        'accuracy without feature:', accuracy_without,
        'difference:', difference
    )

lead_source accuracy without feature: 0.642 difference: 0.0030000000000000027
industry accuracy without feature: 0.645 difference: 0.0
employment_status accuracy without feature: 0.645 difference: 0.0
location accuracy without feature: 0.645 difference: 0.0
annual_income accuracy without feature: 0.724 difference: -0.07899999999999996
number_of_courses_viewed accuracy without feature: 0.643 difference: 0.0020000000000000018
interaction_count accuracy without feature: 0.601 difference: 0.04400000000000004
lead_score accuracy without feature: 0.644 difference: 0.0010000000000000009


In [43]:
# Question 6: Parameter tuning

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = (y_pred == y_val).mean()

    print(f"C={C}, accuracy={accuracy:.3f}")

C=1e-06, accuracy=0.598
C=1e-05, accuracy=0.598
C=0.0001, accuracy=0.613
C=0.001, accuracy=0.645
